# Padronizando colunas e corrigindo tipos

## Task 2: padronizar os nomes das colunas e corrigir os tipos

## O problema

- O público chega como texto, com vírgula de milhar (`"62,576"`), e a data como texto num formato próprio
  (`"13 Jun 1950 - 12:00"`).
- As colunas têm espaços (`Home Team Name`).

Objetivo: nomes simples, em português, e cada coluna com o tipo certo.

In [1]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-padronizacao-partidas")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

Começamos de onde a task 1 parou: o código do fim do vídeo 2.1.

In [2]:
from wc_pipeline.schemas import MATCHES

partidas_brutas = (
    spark.read
    .option("header", "true")
    .schema(MATCHES)
    .csv(f"{RAW}/WorldCupMatches.csv")
)

partidas_limpas = (
    partidas_brutas
    .filter(F.col("MatchID").isNotNull())
    .dropDuplicates(["MatchID"])
)

## select: escolher e renomear colunas

`select` monta um DataFrame só com as colunas pedidas; `.alias()` dá o nome novo.

In [3]:
partidas = partidas_limpas.select(
    F.col("Year").alias("ano"),
    F.col("MatchID").alias("match_id"),
    F.col("Stage").alias("fase"),
    F.col("Home Team Name").alias("mandante"),
    F.col("Home Team Goals").alias("gols_mandante"),
    F.col("Away Team Goals").alias("gols_visitante"),
    F.col("Away Team Name").alias("visitante"),
    F.col("Attendance").alias("publico"),
    F.col("Datetime").alias("data_partida"),
)

partidas.show(5)

+----+--------+-----------+--------+-------------+--------------+---------+-------+-------------------+
| ano|match_id|       fase|mandante|gols_mandante|gols_visitante|visitante|publico|       data_partida|
+----+--------+-----------+--------+-------------+--------------+---------+-------+-------------------+
|1950|    1001|Round of 16| England|            2|             1|    Italy| 62,576|13 Jun 1950 - 12:00|
|1950|    1002|Semi-finals|   Italy|            5|             3|  Germany| 36,643|14 Jun 1950 - 18:00|
|1950|    1003|    Group A| England|            2|             3| Portugal| 23,392|15 Jun 1950 - 19:00|
|1950|    1004|      Final|   Spain|            1|             0|   Mexico| 82,391|16 Jun 1950 - 01:00|
|1950|    1005|    Group A|  France|            4|             4| Portugal| 71,498|16 Jun 1950 - 20:00|
+----+--------+-----------+--------+-------------+--------------+---------+-------+-------------------+
only showing top 5 rows


`F.col("Year")` aponta para uma coluna e aceita nomes com espaço.

## withColumn: corrigindo o público

`withColumn("nome", expressão)` cria uma coluna, ou substitui uma que já existe. Para mudar o tipo, `cast`:

In [4]:
partidas.withColumn("publico", F.col("publico").cast("int")).show(3)

{"ts": "2026-10-21 18:51:09.994", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '62,576' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 1 in cell [5]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o94.showString.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '62,576' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 1 in cell [5]\n\n\tat org.apache.spark.sql.errors.QueryExecutionError

NumberFormatException: [CAST_INVALID_INPUT] The value '62,576' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 1 in cell [5]


`regexp_replace` troca um texto por outro dentro da coluna:

In [5]:
partidas = partidas.withColumn(
    "publico",
    F.regexp_replace("publico", ",", "").cast("int"),
)

partidas.select("mandante", "visitante", "publico").show(3)

+--------+---------+-------+
|mandante|visitante|publico|
+--------+---------+-------+
| England|    Italy|  62576|
|   Italy|  Germany|  36643|
| England| Portugal|  23392|
+--------+---------+-------+
only showing top 3 rows


## withColumn: corrigindo a data

Formato `"13 Jun 1950 - 12:00"` = `"dd MMM yyyy - HH:mm"`. Converte com `to_timestamp`:

In [6]:
FORMATO_DATA = "dd MMM yyyy - HH:mm"

partidas = partidas.withColumn(
    "data_partida",
    F.to_timestamp("data_partida", FORMATO_DATA),
)

partidas.select("mandante", "visitante", "data_partida").show(3)

+--------+---------+-------------------+
|mandante|visitante|       data_partida|
+--------+---------+-------------------+
| England|    Italy|1950-06-13 12:00:00|
|   Italy|  Germany|1950-06-14 18:00:00|
| England| Portugal|1950-06-15 19:00:00|
+--------+---------+-------------------+
only showing top 3 rows


## Duas colunas novas

`total_gols` e `resultado` (`F.when(...).otherwise(...)`, o `CASE WHEN` do SQL). O resultado fica em `partidas`.

In [7]:
partidas = (
    partidas
    .withColumn("total_gols", F.col("gols_mandante") + F.col("gols_visitante"))
    .withColumn(
        "resultado",
        F.when(F.col("gols_mandante") > F.col("gols_visitante"), "mandante")
         .when(F.col("gols_mandante") < F.col("gols_visitante"), "visitante")
         .otherwise("empate"),
    )
)

partidas.printSchema()
partidas.select("ano", "mandante", "visitante", "total_gols", "resultado", "publico", "data_partida").show(5)

root
 |-- ano: integer (nullable = true)
 |-- match_id: integer (nullable = true)
 |-- fase: string (nullable = true)
 |-- mandante: string (nullable = true)
 |-- gols_mandante: integer (nullable = true)
 |-- gols_visitante: integer (nullable = true)
 |-- visitante: string (nullable = true)
 |-- publico: integer (nullable = true)
 |-- data_partida: timestamp (nullable = true)
 |-- total_gols: integer (nullable = true)
 |-- resultado: string (nullable = false)

+----+--------+---------+----------+---------+-------+-------------------+
| ano|mandante|visitante|total_gols|resultado|publico|       data_partida|
+----+--------+---------+----------+---------+-------+-------------------+
|1950| England|    Italy|         3| mandante|  62576|1950-06-13 12:00:00|
|1950|   Italy|  Germany|         8| mandante|  36643|1950-06-14 18:00:00|
|1950| England| Portugal|         5|visitante|  23392|1950-06-15 19:00:00|
|1950|   Spain|   Mexico|         1| mandante|  82391|1950-06-16 01:00:00|
|1950|  Fr

## Conferindo os vazios

Quantas partidas ficaram sem público ou sem data? A mesma contagem do vídeo 2.1: `filter` com `isNull()` e `count()`.

In [8]:
print("partidas    :", partidas.count())
print("público nulo:", partidas.filter(F.col("publico").isNull()).count())
print("data nula   :", partidas.filter(F.col("data_partida").isNull()).count())

partidas    : 562
público nulo: 0
data nula   : 0


## Para consultar depois

- **Três jeitos de se referir a uma coluna:** `"ano"`, `F.col("ano")` e `jogos.ano`. Os três funcionam; o `F.col` é o
  mais robusto.
- **`try_cast` e `try_to_timestamp`:** fazem a mesma conversão, mas devolvem `null` em vez de erro quando o valor é
  inválido. Se usar, conte os nulos depois, para o problema não passar em silêncio.
- **`filter` com várias condições:** `&` (e), `|` (ou) e `~` (não), com cada condição entre parênteses, por exemplo
  `(F.col("ano") >= 1990) & (F.col("total_gols") > 5)`.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [9]:
spark.stop()